In [1]:
from emulator.interpolator import util
from astropy.constants import sigma_sb, L_sun
import astropy.units as u
import numpy as np

def blackbody_L(t_exp, T, v):
    r = v*t_exp
    L = 4*np.pi*r**2 * sigma_sb * T**4
    return L.to(u.erg/u.s)

T = 5500 * u.K
v = 6000 * u.km/u.s
t_min = 10*u.day
t_max = 23*u.day
v_min = 6000*u.km/u.s
v_max = 7000*u.km/u.s

print('Best fit = 8.237')
L_min = blackbody_L(t_min, T, v_min)
L_max = blackbody_L(t_max, T, v_max)
print(t_min, ':', util.to_loglsun(L_min))
print(t_max, ':', util.to_loglsun(L_max))

L = util.to_lum(8.334207183569124)
v_phot = 7609.0866464947985 * u.km/u.s
t_exp = 17.499252278542855 * u.day
R = v_phot * t_exp
T = (L/(4*np.pi*R**2*sigma_sb))**(1/4)
print(T.to(u.K))

Best fit = 8.237
10.0 d : 7.660630415257754
23.0 d : 8.517979666554165
5440.721501290067 K


In [ ]:
# {'log_lsun': 8.26127818193484, 't_exp': 14.422701074736008, 'v_start': 6299.351682158407, 'X': 0.8274156670226834, 'Z': 0.021898409649868618, 'n': 10.999786944928651}
v_start = util.calc_v_phot(t_exp=14.2270107*u.day, X=0.8274156670226834, n=10.99978)
print(v_start)

8322.851637047681 km / s


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats

# Obtain samples
param_names = ['log_lsun', 't_exp', 'v_start', 'X', 'Z', 'n']
limits = [(8, 9), (10, 23), (5600, 7000), (0.55, 0.95), (5e-4, 5e-2), (8, 14)]
n = 10000

distribs = []
for min, max in limits:
    distribs.append(scipy.stats.uniform(min, max - min))

def my_prior_transform(cube):
    params = [distribs[i].ppf(cube[i]) for i in range(len(cube))]
    return params

# Create DataFrame
data = {}
for param in param_names:
    data[param] = []
    data[param + '_u'] = []

for i in range(n):
    cube = np.random.rand(len(param_names))
    params = my_prior_transform(cube)
    for i, value in enumerate(params):
        param = param_names[i]
        data[param].append(value)
        data[param + '_u'].append(cube[i])

df = pd.DataFrame(data)
df.index.name = 'id'
df.to_csv('grid.csv')

In [ ]:
import pandas as pd
import numpy as np

# Seed samples for noise variation tests
n = 1000
seeds = np.random.randint(low=10000000, high=100000000, size=n)
virtual_packets = [3]*int(n/2) + [6]*int(n/2)

df = pd.DataFrame({'seed': seeds, 'virtual_packets': virtual_packets})
df.index.name = 'id'
df.to_csv('grid.csv')
